# Teste 2:  Análise do retriever de querys (Ensemble Retriever de: MMR + BM25)

Esse notebook foi utilizado para testar algumas variáveis baseadas na busca que foi implementada no RAG.
* O que analisa?

1. MMR -> Quantidade = K ; Quantidade de documentos analisados = fetch_k ; Multiplicador_lambda = lambda_mult. Ele principalmente vai observar a query do usuário e se está vindo chunks bons
2. BM25 -> Quantidade = K. Mesma coisa do MMR
3. EnsembleRetriever -> Será que alterar os pesos podem retornar coisas mais valiosas? bom ponto a analisar

## 1.1 Instalar as Dependências

In [2]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q langchain-huggingface sentence-transformers
!pip install -q chromadb pymupdf rank-bm25
!pip install -q langchain-classic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0

In [3]:
!pip list | grep langchain

langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.7
langchain-community                      0.4.1
langchain-core                           1.3.3
langchain-huggingface                    1.2.2
langchain-protocol                       0.0.15
langchain-text-splitters                 1.1.2


In [4]:
!pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 8.2 MB/s eta 0:00:00


In [5]:
# faz o download do livro pelo notebook python do Colab
!wget https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_experimental.text_splitter import SemanticChunker
import torch
import re

arquivo = 'jk_couto_2ed.pdf'
loader = PyMuPDFLoader(f"/content/{arquivo}")
all_pages = loader.load()

print(f"Total de páginas carregadas: {len(all_pages)}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f" Using device: {device}")

--2026-05-08 02:24:11--  https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf
Resolving domainpublic.wordpress.com (domainpublic.wordpress.com)... 192.0.78.13, 192.0.78.12
Connecting to domainpublic.wordpress.com (domainpublic.wordpress.com)|192.0.78.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16321321 (16M) [application/pdf]
Saving to: ‘jk_couto_2ed.pdf.1’

jk_couto_2ed.pdf.1  100%[===================>]  15.56M  --.-KB/s    in 0.08s   

2026-05-08 02:24:11 (187 MB/s) - ‘jk_couto_2ed.pdf.1’ saved [16321321/16321321]

Total de páginas carregadas: 473
 Using device: cuda


## 1.2 Criando a KB

In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": device}
)

text_splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85
)

chunks = text_splitter.split_documents(all_pages)
print(f"Foram criados {len(chunks)} chunks")

vector_store = Chroma.from_documents(chunks, embeddings_model)
print("Base vetorial criada!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Foram criados 1705 chunks
Base vetorial criada!


In [ ]:
query = """Qual foi o papel de Juscelino Kubitschek na construção de Brasília e como isso impactou
o desenvolvimento econômico do Brasil durante seu governo?"""

## Teste 1: MMR (Maximal Marginal Relevance)

Acredito que manter o lambda_mult em 0.6 seria o ideal, por isso deixei ele menos variável nessa parte, porém acredito que o que mais vai afetar é o K e o fetch_K. Vamos lá

In [ ]:
mmr_configs = [
    {'k': 3, 'fetch_k': 20, 'lambda_mult': 0.5},
    {'k': 5, 'fetch_k': 20, 'lambda_mult': 0.6},
    {'k': 5, 'fetch_k': 30, 'lambda_mult': 0.6},
    {'k': 8, 'fetch_k': 30, 'lambda_mult': 0.6},
    {'k': 10, 'fetch_k': 40, 'lambda_mult': 0.7},
]

for config in mmr_configs:
    print(f"\n{'='*60}")
    print(f"MMR: k={config['k']}, fetch_k={config['fetch_k']}, lambda={config['lambda_mult']}")
    print(f"{'='*60}")

    retriever = vector_store.as_retriever(search_type="mmr", search_kwargs=config)
    docs = retriever.invoke(query)

    print(f"Recuperou (retrieved): {len(docs)} docs")
    for i, doc in enumerate(docs):
        print(f"\n[{i+1}] {len(doc.page_content)} chars")
        print(doc.page_content[:200] + "...")


MMR: k=3, fetch_k=20, lambda=0.5
Recuperou (retrieved): 3 docs

[1] 864 chars
JK • 267
“Nos sete anos de trabalho em Brasília, guardo do 
presidente Juscelino Kubitschek apenas a lembrança 
de um homem cheio de entusiasmo, desejoso de 
fazer qualquer coisa importante para este ...

[2] 370 chars
ed. Brasília: Senado 
Federal, 1994. BADARÓ, Murilo Paulino. José Maria Alkmim: uma biografia. Rio de Janeiro: Nova 
Fronteira, 1996. BAER, Werner. A industrialização e o desenvolvimento econômico no ...

[3] 273 chars
O 
mercado de trabalho era estreitíssimo. Juscelino, sobre a pequenina 
Belo Horizonte de então: “Como dizia um amigo meu: um arraial com 
bonde elétrico.”. O Brasil era essencialmente agrícola, uma e...

MMR: k=5, fetch_k=20, lambda=0.6
Recuperou (retrieved): 5 docs

[1] 864 chars
JK • 267
“Nos sete anos de trabalho em Brasília, guardo do 
presidente Juscelino Kubitschek apenas a lembrança 
de um homem cheio de entusiasmo, desejoso de 
fazer qualquer coisa importante para este 

Minhas impressões é que chegou num ponto em que não houve muita discrepância, porém aumentar o fetch_k para 30 melhorou um pouco no conteúdo um pouco mais diverso entre os chunks recolhidos.

## Teste 2: BM25 (Keyword-based)

In [ ]:
bm25_ks = [3, 5, 8, 10, 15]

for k in bm25_ks:
    print(f"\n{'='*60}")
    print(f"BM25: k={k}")
    print(f"{'='*60}")

    bm25 = BM25Retriever.from_documents(chunks, k=k)
    docs = bm25.invoke(query)

    print(f"Recuperou (retrieved): {len(docs)} docs")
    for i, doc in enumerate(docs):
        print(f"\n[{i+1}] {len(doc.page_content)} chars")
        print(doc.page_content[:200] + "...")


BM25: k=3
Recuperou (retrieved): 3 docs

[1] 735 chars
JK • 269
“Nenhum desses governos foi tão cheio de 
consequências quanto o seu. A construção de Brasília 
e a conquista do oeste desviaram completamente 
o curso de nossa história e deram-lhe perspecti...

[2] 1075 chars
Era o que todos queriam saber. O sonho maior de Goiás e de quase todo o Brasil profundo. Por que o primeiro comício no quase vazio goiano, de complicado 
acesso e escassos eleitores? Por que não Belo ...

[3] 630 chars
Mas não se passava disso. Não havia – pelo 
menos não me lembro de ter havido durante o tempo de minha forma­
ção – a convicção de que o Brasil necessitava desenvolver-se para sobre­
viver, de que se ...

BM25: k=5
Recuperou (retrieved): 5 docs

[1] 735 chars
JK • 269
“Nenhum desses governos foi tão cheio de 
consequências quanto o seu. A construção de Brasília 
e a conquista do oeste desviaram completamente 
o curso de nossa história e deram-lhe perspecti...

[2] 1075 chars
Era o que todos queriam s

Chegou num ponto em que a partir da 5 pra cima, os resultados começaram a ficar menos correspondentes à mensagem, alguma citação já fazia pegar o chunking como todo. Puxaram até chunks com mais de 1000 caracteres... Visualmente falando, os 5 melhores seriam o suficiente

## Teste 3: Ensemble (BM25 + MMR)

Aqui eu quis testar como seria ter um peso maior para algum tipo de busca (nesse caso, o BM25), o que acarreteria em melhores respostas? não sei, vamos testar.

In [ ]:
ensemble_configs = [
    {'bm25': 0.3, 'mmr': 0.7},
    {'bm25': 0.5, 'mmr': 0.5},
    {'bm25': 0.6, 'mmr': 0.4},
    {'bm25': 0.4, 'mmr': 0.6},
]

for config in ensemble_configs:
    print(f"\n{'='*60}")
    print(f"Ensemble: BM25={config['bm25']}, MMR={config['mmr']}")
    print(f"{'='*60}")

    bm25 = BM25Retriever.from_documents(chunks, k=5)
    mmr = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={'k': 5, 'fetch_k': 30, 'lambda_mult': 0.6}
    )

    ensemble = EnsembleRetriever(
        retrievers=[bm25, mmr],
        weights=[config['bm25'], config['mmr']]
    )

    docs = ensemble.invoke(query)

    print(f"Retrieved: {len(docs)} docs")
    for i, doc in enumerate(docs[:5]):
        print(f"\n[{i+1}] {len(doc.page_content)} chars")
        print(doc.page_content[:200] + "...")


Ensemble: BM25=0.3, MMR=0.7
Retrieved: 10 docs

[1] 864 chars
JK • 267
“Nos sete anos de trabalho em Brasília, guardo do 
presidente Juscelino Kubitschek apenas a lembrança 
de um homem cheio de entusiasmo, desejoso de 
fazer qualquer coisa importante para este ...

[2] 370 chars
ed. Brasília: Senado 
Federal, 1994. BADARÓ, Murilo Paulino. José Maria Alkmim: uma biografia. Rio de Janeiro: Nova 
Fronteira, 1996. BAER, Werner. A industrialização e o desenvolvimento econômico no ...

[3] 896 chars
Como o garoto pobre de Diamantina 
pôde conquistar o governo de Minas 
e a Presidência da República? Como 
conseguiu revolucionar o desenvol­
vimento brasileiro e inserir o país 
na modernidade sem se...

[4] 273 chars
O 
mercado de trabalho era estreitíssimo. Juscelino, sobre a pequenina 
Belo Horizonte de então: “Como dizia um amigo meu: um arraial com 
bonde elétrico.”. O Brasil era essencialmente agrícola, uma e...

[5] 636 chars
Quem o faria? Mas fez mais do que qualquer outro para consoli

Sinceramente... achei que as respostas advindas pelo MMR foram melhores, acredito que tanto a KB quanto esse tipo de busca é por semântica, porém vou deixar igualado por achar que ambos são igualmente importante pro re-ranking.